<a href="https://colab.research.google.com/github/vishnupriya1002/An-Automated-student-Analysis-System-with-CGPA-Computation-using-Predictive-Analysis-Algorithm/blob/main/Vishnupriya_T_612923101122_Agentic_CAG_Loan_Risk_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI-Based Loan Eligibility and Credit Risk Assessment System
**Student:** Vishnupriya T (612923101122)  
**Platform:** Google Colab  
**Approach:** Machine Learning + Context-Augmented Generation (CAG) + Agentic AI

## Project overview
This notebook demonstrates a loan-risk decision-support prototype. A machine-learning model estimates a risk category from applicant financial features. A context layer adds transparent assessment rules and definitions, and an AI explanation agent turns the structured evidence into an understandable report.

**Responsible-use note:** This is an academic prototype using synthetic demonstration data. It must not be used to approve, reject, price, or rank real loan applicants. The model is not validated for lending use, and protected characteristics must not be used as decision features. Real lending decisions require lawful, validated procedures, fairness testing, explainability, security, and qualified human review.

## 1. Install required libraries

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn joblib gradio google-genai

## 2. Import libraries and configure the notebook

In [ ]:
import os, json, getpass, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.inspection import permutation_importance
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
print("Environment ready")

## 3. Create or upload a dataset

For a reproducible first run, this cell creates a **synthetic educational dataset**. You may instead upload a CSV with the documented feature columns. The generated target is a simulated risk label, not a real-world lending outcome.

In [ ]:
from google.colab import files

USE_SYNTHETIC_DATA = True  # Change to False to upload your own CSV

def make_demo_dataset(n=2500, seed=42):
    rng = np.random.default_rng(seed)
    income = rng.normal(65000, 22000, n).clip(12000, 250000)
    loan_amount = rng.normal(18000, 9000, n).clip(1000, 90000)
    employment_years = rng.integers(0, 31, n)
    debt_ratio = rng.beta(2.2, 5.0, n).clip(0.01, 0.95)
    credit_history_years = rng.gamma(2.5, 2.2, n).clip(0, 30)
    prior_delinquencies = rng.poisson(0.45, n).clip(0, 6)
    existing_loans = rng.poisson(1.1, n).clip(0, 7)
    savings = (income * rng.uniform(0.05, 0.9, n)).clip(0, 150000)
    purpose = rng.choice(["education", "home_improvement", "vehicle", "medical", "business", "personal"], n)
    # Simulated risk score; not a lending policy or validated credit score.
    logit = (-1.6 + 3.4*debt_ratio + 0.45*prior_delinquencies +
             0.18*existing_loans - 0.000018*income -
             0.035*credit_history_years - 0.000008*savings -
             0.025*employment_years + 0.000012*loan_amount)
    probability = 1/(1+np.exp(-logit))
    bad_event = rng.binomial(1, probability, n)
    risk = np.where(bad_event == 1, "higher_risk", "lower_risk")
    return pd.DataFrame({
        "annual_income": income.round(2),
        "loan_amount": loan_amount.round(2),
        "employment_years": employment_years,
        "debt_to_income_ratio": debt_ratio.round(3),
        "credit_history_years": credit_history_years.round(1),
        "prior_delinquencies": prior_delinquencies,
        "existing_loans": existing_loans,
        "savings": savings.round(2),
        "loan_purpose": purpose,
        "risk_label": risk
    })

if USE_SYNTHETIC_DATA:
    df = make_demo_dataset()
else:
    uploaded = files.upload()
    csv_name = next(iter(uploaded))
    df = pd.read_csv(csv_name)

print("Dataset shape:", df.shape)
display(df.head())

## 4. Understand and validate the data

In [ ]:
REQUIRED_COLUMNS = [
    "annual_income", "loan_amount", "employment_years",
    "debt_to_income_ratio", "credit_history_years",
    "prior_delinquencies", "existing_loans", "savings",
    "loan_purpose", "risk_label"
]
missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

df = df[REQUIRED_COLUMNS].copy()
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))
print("Target distribution:")
display(df["risk_label"].value_counts(dropna=False))
print("Numeric summary:")
display(df.describe().T)

## 5. Exploratory data analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="risk_label")
plt.title("Distribution of simulated risk labels")
plt.xlabel("Risk label")
plt.ylabel("Applicant count")
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="annual_income", y="loan_amount", hue="risk_label", alpha=.55)
plt.title("Income versus requested loan amount")
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(data=df, x="debt_to_income_ratio", hue="risk_label", bins=25, element="step")
plt.title("Debt-to-income ratio distribution")
plt.show()

## 6. Preprocess data and train the risk model

In [ ]:
X = df.drop(columns=["risk_label"])
y = df["risk_label"]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250, max_depth=10, min_samples_leaf=4,
        class_weight="balanced", random_state=RANDOM_STATE
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Test accuracy (synthetic data only):", round(accuracy_score(y_test, predictions), 3))
print(classification_report(y_test, predictions))

## 7. Evaluate model behaviour

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, predictions, labels=labels)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.show()

print("Interpretation: metrics describe performance on a held-out split of synthetic data only.")

## 8. Inspect influential features

In [ ]:
perm = permutation_importance(
    model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE,
    scoring="accuracy"
)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values()
display(importance.to_frame("mean_permutation_importance"))
importance.plot(kind="barh", figsize=(8,5), title="Permutation importance (test split)")
plt.xlabel("Mean decrease in accuracy when feature is shuffled")
plt.show()

## 9. Define transparent assessment context

The following context is illustrative and is included to explain the model output. It is **not** a bank's underwriting policy, regulatory guidance, or a substitute for a lender's approved criteria. The thresholds below are configurable demonstration thresholds only.

In [ ]:
ASSESSMENT_CONTEXT = {
    "purpose": "Educational decision-support explanation, not a credit decision.",
    "definitions": {
        "debt_to_income_ratio": "Monthly debt obligations divided by gross monthly income. This dataset stores a simplified ratio value.",
        "higher_risk": "The synthetic training label associated with a higher simulated adverse-event likelihood.",
        "lower_risk": "The synthetic training label associated with a lower simulated adverse-event likelihood."
    },
    "demo_guidance": [
        "A high debt-to-income ratio can indicate less monthly repayment capacity.",
        "Stable income and a longer employment history may provide useful context, but do not guarantee repayment.",
        "Prior delinquencies and existing debt may be relevant risk indicators and should be verified.",
        "Savings may provide a financial buffer but must not be treated as proof of eligibility.",
        "A model prediction is uncertain and must be reviewed with verified documents and applicable lender policy."
    ],
    "demo_thresholds": {
        "debt_to_income_ratio_attention": 0.40,
        "minimum_income_for_demo_flag": 20000,
        "high_delinquency_count": 2
    }
}
print(json.dumps(ASSESSMENT_CONTEXT, indent=2))

## 10. Applicant input schema and validation

In [ ]:
APPLICANT_FIELDS = [
    "annual_income", "loan_amount", "employment_years",
    "debt_to_income_ratio", "credit_history_years",
    "prior_delinquencies", "existing_loans", "savings", "loan_purpose"
]

def validate_applicant(applicant):
    missing = [k for k in APPLICANT_FIELDS if k not in applicant]
    if missing:
        raise ValueError(f"Missing applicant fields: {missing}")
    numeric = APPLICANT_FIELDS[:-1]
    for key in numeric:
        applicant[key] = float(applicant[key])
        if applicant[key] < 0:
            raise ValueError(f"{key} cannot be negative")
    if not 0 <= applicant["debt_to_income_ratio"] <= 1.5:
        raise ValueError("debt_to_income_ratio must be between 0 and 1.5")
    applicant["employment_years"] = int(applicant["employment_years"])
    applicant["prior_delinquencies"] = int(applicant["prior_delinquencies"])
    applicant["existing_loans"] = int(applicant["existing_loans"])
    return applicant

## 11. Agent tools: prediction, affordability context, and evidence gathering

In [ ]:
def predict_risk(applicant):
    row = pd.DataFrame([applicant])[X.columns]
    predicted = model.predict(row)[0]
    probabilities = model.predict_proba(row)[0]
    classes = model.named_steps["classifier"].classes_
    probs = {str(c): float(p) for c, p in zip(classes, probabilities)}
    return {"predicted_risk": str(predicted), "class_probabilities": probs}

def calculate_demo_indicators(applicant):
    # Simple context indicators only; not an affordability or approval calculation.
    flags = []
    if applicant["debt_to_income_ratio"] >= ASSESSMENT_CONTEXT["demo_thresholds"]["debt_to_income_ratio_attention"]:
        flags.append("Debt-to-income ratio reaches the illustrative attention threshold.")
    if applicant["annual_income"] < ASSESSMENT_CONTEXT["demo_thresholds"]["minimum_income_for_demo_flag"]:
        flags.append("Annual income is below the illustrative demo flag threshold.")
    if applicant["prior_delinquencies"] >= ASSESSMENT_CONTEXT["demo_thresholds"]["high_delinquency_count"]:
        flags.append("Prior delinquency count reaches the illustrative attention threshold.")
    if not flags:
        flags.append("No configured demo attention flag was triggered; this does not establish eligibility.")
    return flags

def retrieve_context(applicant, prediction, flags):
    return {
        "definitions": ASSESSMENT_CONTEXT["definitions"],
        "guidance": ASSESSMENT_CONTEXT["demo_guidance"],
        "prediction": prediction,
        "attention_flags": flags,
        "applicant_facts": applicant,
        "limitations": [
            "Synthetic training data; not representative of a lender's portfolio.",
            "No identity, employment, income, or credit-bureau verification.",
            "No protected attributes are collected or used by this notebook.",
            "Do not use the output as an approval, rejection, pricing, or ranking decision."
        ]
    }

## 12. Agentic CAG orchestration

The orchestrator follows a controlled sequence: validate input → call the risk model → run transparent rule checks → retrieve relevant context → generate an evidence-grounded explanation. The agent does not have permission to approve or reject a loan.

In [ ]:
def run_cag_agent(applicant):
    applicant = validate_applicant(dict(applicant))
    prediction = predict_risk(applicant)
    flags = calculate_demo_indicators(applicant)
    context = retrieve_context(applicant, prediction, flags)
    return {
        "status": "decision_support_only",
        "prediction": prediction,
        "attention_flags": flags,
        "context": context
    }

## 13. Configure Gemini for natural-language explanations (optional)

Gemini is used only to explain the structured evidence produced by the notebook. It must not invent missing facts or convert the model output into a lending decision. Add your API key to **Colab Secrets** with the name `GEMINI_API_KEY` (recommended), or enter it in the hidden prompt below. Never print or share the key.

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = None
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("Enter Gemini API key (input hidden; optional): ").strip()

gemini_client = None
if GEMINI_API_KEY:
    from google import genai
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client configured.")
else:
    print("Gemini not configured. The template-based explanation will be used.")

## 14. Generate a context-grounded assessment report

In [ ]:
def template_explanation(agent_result):
    p = agent_result["prediction"]
    facts = agent_result["context"]["applicant_facts"]
    flags = agent_result["attention_flags"]
    return f'''## Applicant risk-assessment summary

**Model output:** `{p["predicted_risk"]}`
**Class probabilities:** {json.dumps({k: round(v, 3) for k,v in p["class_probabilities"].items()})}

### Applicant context
- Annual income: {facts["annual_income"]:,.0f}
- Requested loan amount: {facts["loan_amount"]:,.0f}
- Debt-to-income ratio supplied: {facts["debt_to_income_ratio"]:.1%}
- Employment history: {facts["employment_years"]} years
- Credit history length: {facts["credit_history_years"]} years

### Attention indicators
{chr(10).join("- " + f for f in flags)}

### Interpretation
This is a prediction from a model trained on synthetic demonstration data. It is not a finding of eligibility, creditworthiness, or repayment ability.

### Recommended human-review checks
Verify income and employment documents, confirm liabilities and credit-history information through authorized sources, apply the lender's current approved policy, and document the reasons for any final decision.

**Important:** Do not approve or reject an application based only on this prototype.
'''

def generate_explanation(agent_result):
    if gemini_client is None:
        return template_explanation(agent_result)
    prompt = f'''
You are an explanation assistant for an academic loan-risk decision-support prototype.
Use ONLY the JSON evidence below. Do not infer missing facts, invent regulations,
give an approval/rejection, recommend an interest rate, or claim the model is validated.
Explain model uncertainty and limitations. State clearly that a qualified human must
make any real lending decision. Use concise headings and plain language.

EVIDENCE JSON:
{json.dumps(agent_result["context"], indent=2)}
'''
    try:
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return response.text
    except Exception as e:
        print("Gemini request failed; using local explanation. Error type:", type(e).__name__)
        return template_explanation(agent_result)

## 15. Run a sample applicant

In [ ]:
sample_applicant = {
    "annual_income": 72000,
    "loan_amount": 22000,
    "employment_years": 5,
    "debt_to_income_ratio": 0.32,
    "credit_history_years": 6,
    "prior_delinquencies": 0,
    "existing_loans": 1,
    "savings": 12000,
    "loan_purpose": "vehicle"
}
sample_result = run_cag_agent(sample_applicant)
display(Markdown(generate_explanation(sample_result)))

## 16. Interactive Gradio application

In [ ]:
import gradio as gr

def assess_applicant_ui(annual_income, loan_amount, employment_years,
                        debt_ratio, credit_history_years, delinquencies,
                        existing_loans, savings, loan_purpose):
    applicant = {
        "annual_income": annual_income,
        "loan_amount": loan_amount,
        "employment_years": employment_years,
        "debt_to_income_ratio": debt_ratio,
        "credit_history_years": credit_history_years,
        "prior_delinquencies": delinquencies,
        "existing_loans": existing_loans,
        "savings": savings,
        "loan_purpose": loan_purpose
    }
    result = run_cag_agent(applicant)
    report = generate_explanation(result)
    return report, json.dumps(result["prediction"], indent=2)

with gr.Blocks(title="Agentic CAG Loan Risk Assessment") as demo:
    gr.Markdown("# Agentic AI-Based Loan Eligibility and Credit Risk Assessment")
    gr.Markdown("Academic decision-support demonstration only. Not for real lending decisions.")
    with gr.Row():
        annual_income = gr.Number(label="Annual income", value=72000, minimum=0)
        loan_amount = gr.Number(label="Requested loan amount", value=22000, minimum=0)
        employment_years = gr.Number(label="Employment years", value=5, minimum=0, precision=0)
    with gr.Row():
        debt_ratio = gr.Number(label="Debt-to-income ratio (0–1.5)", value=0.32, minimum=0, maximum=1.5)
        credit_history_years = gr.Number(label="Credit history (years)", value=6, minimum=0)
        savings = gr.Number(label="Savings", value=12000, minimum=0)
    with gr.Row():
        delinquencies = gr.Number(label="Prior delinquencies", value=0, minimum=0, precision=0)
        existing_loans = gr.Number(label="Existing loans", value=1, minimum=0, precision=0)
        loan_purpose = gr.Dropdown(
            ["education", "home_improvement", "vehicle", "medical", "business", "personal"],
            value="vehicle", label="Loan purpose"
        )
    run_btn = gr.Button("Analyse applicant context")
    report_out = gr.Markdown()
    prediction_out = gr.Code(label="Structured model output", language="json")
    run_btn.click(
        assess_applicant_ui,
        inputs=[annual_income, loan_amount, employment_years, debt_ratio,
                credit_history_years, delinquencies, existing_loans, savings, loan_purpose],
        outputs=[report_out, prediction_out]
    )

demo.launch(share=True, debug=False)

## 17. Save the trained model and assessment context

In [ ]:
import joblib
joblib.dump(model, "loan_risk_demo_model.joblib")
with open("assessment_context.json", "w") as f:
    json.dump(ASSESSMENT_CONTEXT, f, indent=2)
print("Saved model and context files.")

## 18. Download project artifacts

In [ ]:
from google.colab import files
files.download("loan_risk_demo_model.joblib")
files.download("assessment_context.json")

## 19. Testing checklist
- [ ] Run all cells from top to bottom in a fresh Colab runtime.
- [ ] Confirm the dataset has the required columns and target labels.
- [ ] Test valid and invalid applicant inputs.
- [ ] Confirm the output includes prediction probabilities and attention flags.
- [ ] Test the local explanation with Gemini disabled.
- [ ] If Gemini is enabled, confirm the explanation uses only supplied evidence.
- [ ] Test the Gradio interface with multiple sample profiles.
- [ ] Record model metrics and limitations accurately in the project report.

## 20. Limitations and future enhancements
**Limitations:** synthetic labels, limited features, no real credit-bureau integration, no document verification, no temporal validation, no formal fairness audit, and no production security controls.

**Future enhancements:** use a legally obtained and appropriately governed dataset; validate labels and data quality; conduct subgroup fairness and stability assessments where lawful; calibrate probabilities; add secure document extraction with applicant consent; integrate a lender-approved policy knowledge base; log human review and override reasons; and conduct independent validation before any operational use.

## Conclusion
This notebook demonstrates how machine learning, retrieved assessment context, and a controlled AI explanation agent can work together to produce a transparent loan-risk assessment summary. It is an academic prototype and does not make lending decisions.